# Paper PoRT Direct Router Diagnostic From Notebook 30

This notebook reads the full notebook 30 artifact zip and trains/tests offline selectors for the direct `choose initial vs rethink` objective. It does not run the target model again.

The diagnostic uses rows where notebook 30 actually generated both an initial answer and a rethink answer. It compares simple confidence routing against TF-IDF/logistic selectors and reports held-out accuracy with group splits by `domain::row_index` to avoid leaking the same WMDP question across variants.

This is a recreated-artifact diagnostic, not an official paper checkpoint metric.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


In [ ]:
required_packages = {
    'numpy': 'numpy',
    'pandas': 'pandas',
    'sklearn': 'scikit-learn',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


## Runtime Config

Default artifact lookup order:

- `PORT_NOTEBOOK30_RESULTS_ZIP_PATH` if set.
- `PROJECT_ROOT/results/notebook30_full_recreated_raw_inverted_route_results.zip` for local runs.
- `/kaggle/working/notebook30_full_recreated_raw_inverted_route_results.zip` for Kaggle runs.
- `PORT_NOTEBOOK30_RESULTS_ZIP_URL` if set.

If running on Kaggle, put the notebook 30 result zip at `/kaggle/working/notebook30_full_recreated_raw_inverted_route_results.zip` or set `PORT_NOTEBOOK30_RESULTS_ZIP_URL` to a stable URL.

In [ ]:
os.environ.setdefault('PORT_RUN_NAME', 'paper_port_direct_router_diagnostic_from_notebook30')
local_zip = PROJECT_ROOT / 'results' / 'notebook30_full_recreated_raw_inverted_route_results.zip'
if local_zip.exists():
    os.environ.setdefault('PORT_NOTEBOOK30_RESULTS_ZIP_PATH', str(local_zip))
elif IS_KAGGLE:
    os.environ.setdefault('PORT_NOTEBOOK30_RESULTS_ZIP_PATH', '/kaggle/working/notebook30_full_recreated_raw_inverted_route_results.zip')

runtime_keys = [
    'PORT_RUN_NAME',
    'PORT_NOTEBOOK30_RESULTS_ZIP_PATH',
    'PORT_NOTEBOOK30_RESULTS_ZIP_URL',
    'PORT_ROUTER_EVAL_SIZE',
    'PORT_ROUTER_TEST_SIZE',
    'PORT_SEED',
]
print(json.dumps({key: os.environ.get(key) for key in runtime_keys}, indent=2))


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()


def has_project_layout(path):
    path = Path(path)
    return (path / 'PoRT_pipeline' / 'WMDP' / 'port_pipeline_wmdp.py').exists() and (path / 'dataset').exists()


def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()


if 'PROJECT_ROOT' not in globals() or not has_project_layout(PROJECT_ROOT):
    PROJECT_ROOT = clone_or_use_project()
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if 'commit_sha' not in globals():
    commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')

runner_path = PROJECT_ROOT / 'notebooks' / 'common' / 'port_direct_router_diagnostic.py'
if not runner_path.exists():
    raise FileNotFoundError(runner_path)

common_dir = str(runner_path.parent)
if common_dir not in sys.path:
    sys.path.insert(0, common_dir)

spec = importlib.util.spec_from_file_location('port_direct_router_diagnostic', runner_path)
port_direct_router_diagnostic = importlib.util.module_from_spec(spec)
spec.loader.exec_module(port_direct_router_diagnostic)

result = port_direct_router_diagnostic.run(
    project_root=PROJECT_ROOT,
    is_kaggle=IS_KAGGLE,
    commit_sha=commit_sha,
)
print(json.dumps(result, indent=2, default=str))

run_dir = Path(result['artifacts']['summary_json']).parent
for artifact_name in [
    'summary.json',
    'router_model_results.csv',
    'threshold_policy_search.csv',
    'logreg_pre_rethink_threshold_search.csv',
    'logreg_posthoc_initial_rethink_threshold_search.csv',
    'notebook30_route_breakdown_by_job.csv',
]:
    artifact_path = run_dir / artifact_name
    print(f'{artifact_name}: {artifact_path.exists()} {artifact_path}')
